In [ ]:
# Energy features
# Include only melting temperature, GC content and free energy change for guide and target 

from nupack import *
import RNA
import math
import itertools
import numpy as np
from Bio.SeqUtils import MeltingTemp as mt
from Bio.Seq import Seq
import collections

def normalize_guide(value):
    normalized_value = (value - (-12)) / (1 - (-12))
    return normalized_value

def normalize_target(value):
    normalized_value = (value - (-38)) / (-13 - (-38))
    return normalized_value

def normalize_mt(value):
    normalized_value = (value - 5) / (95 - 5)
    return normalized_value

def GCPercent(seq):                       
    counts = collections.Counter(seq)
    percent = (counts['C'] + counts['G'])/float(len(seq)) 
    return percent
    
file_paths = [
    'Feature_MLP_conventional.txt'
]

for file_path in file_paths:
    with open(file_path, 'w') as file:
        pass  

# Define NUPACK model
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

# Load crRNA sequences
file1 = open('iMeta_trans_LbCas12a_sorted_guide_TTTV_filtered.txt', 'r')    
lines = file1.readlines()
guide_array = [line.strip() for line in lines]

# Load target sequences
file2 = open('iMeta_trans_LbCas12a_sorted_target_full_TTTV_filtered.txt', 'r')    
lines = file2.readlines()
full_target_array = [line.strip() for line in lines]
file3 = open('iMeta_trans_LbCas12a_sorted_target_truncated_TTTV_filtered.txt', 'r')    
lines = file3.readlines()
truncated_target_array = [line.strip() for line in lines]

min_guide_energy = np.inf
max_guide_energy = -np.inf
min_target_energy = np.inf
max_target_energy = -np.inf

# Initialize energy_array
energy_array = []

for i in range (0, len(guide_array)):
    energy_array_unit = []

    guide = guide_array[i]
    guide_spacer = guide[-21:]
    target_full = full_target_array[i]
    target_truncated = truncated_target_array[i]

    # Initialize energy_array
    energy_array_guide = []
    energy_array_target = []

    # Compute ensemble energy
    partition_function_guide = pfunc(strands=guide, model=my_model_RNA)
    ensemble_energy_guide = partition_function_guide[1]
    partition_function_target = pfunc(strands=[guide, target_truncated], model=my_model_DNA)
    ensemble_energy_target = partition_function_target[1]
    
    # Compute suboptimal structures and energy
    subopt_structures_guide = subopt(strands=guide, energy_gap=0.1, model=my_model_RNA)  
    subopt_structures_target = subopt(strands=[guide, target_truncated], energy_gap=0.1, model=my_model_DNA)

    # Calculate melting temperature for guide and target in double-stranded form (using Biopython)
    Tm_guide = mt.Tm_NN(guide_spacer, Na=100, Mg=10, nn_table=mt.RNA_NN1)
    Tm_target = mt.Tm_NN(target_truncated, Na=100, Mg=10, nn_table=mt.DNA_NN4)

    # Calculate GC content
    GC_guide = GCPercent(guide_spacer)
    GC_target = GCPercent(target_truncated)
    
    # Calculate/compare subopt_structures_guide[0].energy and subopt_structures_target[0].energy
    if subopt_structures_guide[0].energy + 3.47 > max_guide_energy:
        max_guide_energy = subopt_structures_guide[0].energy + 3.47
    if subopt_structures_guide[0].energy + 3.47 < min_guide_energy:
        min_guide_energy = subopt_structures_guide[0].energy + 3.47
    if subopt_structures_target[0].energy > max_target_energy:
        max_target_energy = subopt_structures_target[0].energy
    if subopt_structures_target[0].energy < min_target_energy:
        min_target_energy = subopt_structures_target[0].energy
        
        
    to_be_added = [normalize_mt(Tm_guide)]
    to_be_added.append(normalize_mt(Tm_target))

    to_be_added.append(GC_guide)
    to_be_added.append(GC_target)
    
    to_be_added.append(normalize_guide(subopt_structures_guide[0].energy + 3.47))
    to_be_added.append(normalize_target(subopt_structures_target[0].energy))

    energy_array_unit.append(to_be_added)

    energy_array_to_append = [energy_array_unit]
    energy_array.append(energy_array_to_append)

energy_array = list(itertools.chain.from_iterable(energy_array))

# Open a file in write mode
with open('Feature_MLP_conventional.txt', 'a') as file:
    # Iterate over each row in the 2D array
    for row in energy_array:
        # Convert each element to a string and join them with spaces
        file.write(' '.join(map(str, row)) + '\n')

print('-------------')

print(min_guide_energy, max_guide_energy, min_target_energy, max_target_energy)